In [42]:
import pandas as pd
import numpy as np

In [43]:
df = pd.read_csv('../Datasets/Nifty_Stocks.csv')

In [44]:
df

,Date,Open,High,Low,Close,Adj Close,Volume,Symbol,Category,Daily_Return,Price_Range,Volatility,Cumulative_Return,Average_Price
0,2024-01-01,3790.000000,3832.000000,3773.000000,3811.100098,3757.422119,825907,TCS,IT_industry,3711.100098,59.000000,3732.999714,NaN,5695.550049
1,2024-01-02,3811.100098,3811.100098,3767.250000,3783.199951,3729.915039,1344068,TCS,IT_industry,3683.199951,43.850098,3711.521697,-0.007321,5702.700073
2,2024-01-03,3767.000000,3771.850098,3687.050049,3691.750000,3639.753174,1803075,TCS,IT_industry,3591.750000,84.800049,3671.977407,-0.031493,5612.875000
3,2024-01-04,3701.750000,3719.000000,3651.000000,3666.800049,3615.154541,3598144,TCS,IT_industry,3566.800049,68.000000,3619.430895,-0.038252,5535.150024
4,2024-01-05,3675.000000,3747.750000,3674.850098,3737.899902,3685.252930,1963127,TCS,IT_industry,3637.899902,72.899902,3649.436771,-0.018862,5543.949951
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10813,2024-09-05,654.000000,684.000000,653.700012,664.500000,664.500000,3457002,RITES,Railways,564.500000,30.299988,585.625280,97.873441,986.250000
10814,2024-09-06,667.799988,679.849976,657.099976,661.849976,661.849976,1482454,RITES,Railways,561.849976,22.750000,580.567661,97.869453,998.724976
10815,2024-09-09,665.750000,677.900024,664.000000,672.750000,672.750000,1991325,RITES,Railways,572.750000,13.900024,579.200656,97.885922,1002.125000
10816,2024-09-10,677.000000,684.000000,675.500000,678.900024,678.900024,624747,RITES,Railways,578.900024,8.500000,584.500814,97.895063,1016.450012


# Phase 1: Data Preparation

### Data Preparation:

#### Handling Null Values:

In [45]:
df.isnull().sum()

Date                 0
Open                 0
High                 0
Low                  0
Close                0
Adj Close            0
Volume               0
Symbol               0
Category             0
Daily_Return         0
Price_Range          0
Volatility           0
Cumulative_Return    1
Average_Price        0
dtype: int64

In [46]:
df.Cumulative_Return =df.Cumulative_Return.fillna(0)

#### Converting Date Column To Its Original Datatype:

In [47]:
df.dtypes

Date                  object
Open                 float64
High                 float64
Low                  float64
Close                float64
Adj Close            float64
Volume                 int64
Symbol                object
Category              object
Daily_Return         float64
Price_Range          float64
Volatility           float64
Cumulative_Return    float64
Average_Price        float64
dtype: object

In [48]:
df.Date = df.Date.astype('datetime64[ns]')

#### Sorting Data By Dates & Symbols:

In [49]:
df = df.sort_values(by=['Date', 'Symbol'])

## Feature Engineering:

In [55]:
df['SMA_50'] = df.Close.rolling(window=50).mean()
df['SMA_200'] = df.Close.rolling(window=200).mean()

In [57]:
delta = df.Close.diff()

gain = delta.where(delta > 0, 0)
loss = -delta.where(delta < 0, 0)

avg_gain = gain.rolling(window=14).mean()
avg_loss = loss.rolling(window=14).mean()

rs = avg_gain / avg_loss
df['RSI_14'] = 100 - (100 / (1 + rs))

In [59]:
ema_12 = df['Close'].ewm(span=12, adjust=False).mean()
ema_26 = df['Close'].ewm(span=26, adjust=False).mean()

df['MACD'] = ema_12 - ema_26

#### Handling Null Values Produced By Feature Engineering:

In [63]:
df.isnull().sum()

Date                 0
Open                 0
High                 0
Low                  0
Close                0
Adj Close            0
Volume               0
Symbol               0
Category             0
Daily_Return         0
Price_Range          0
Volatility           0
Cumulative_Return    0
Average_Price        0
SMA_50               0
SMA_200              0
RSI_14               0
MACD                 0
dtype: int64

In [62]:
df.SMA_50 = df.SMA_50.fillna(0)
df.SMA_200 = df.SMA_200.fillna(0)
df.RSI_14 = df.RSI_14.fillna(0)

In [64]:
df

,Date,Open,High,Low,Close,Adj Close,Volume,Symbol,Category,Daily_Return,Price_Range,Volatility,Cumulative_Return,Average_Price,SMA_50,SMA_200,RSI_14,MACD
4128,2024-01-01,1598.000000,1620.000000,1585.000000,1598.400024,1598.400024,912757,ADANIGREEN,Energy,1498.400024,35.000000,1520.838340,27.097994,2397.200012,0.000000,0.000000,0.000000,0.000000
2236,2024-01-01,1318.000000,1336.900024,1310.000000,1320.099976,1320.099976,460112,AFFLE,IT_industry,1220.099976,26.900024,1237.665116,10.797978,1978.049988,0.000000,0.000000,0.000000,-22.200574
4300,2024-01-01,5741.000000,5765.000000,5707.850098,5750.049805,5736.050781,127030,APOLLOHOSP,HealthCare,5650.049805,57.149902,5665.733902,29.458001,8616.024902,0.000000,0.000000,0.000000,314.045124
4472,2024-01-01,179.050003,180.000000,170.050003,175.100006,174.671494,613446,ARTEMISMED,HealthCare,75.100006,9.949997,82.884068,28.707058,266.600006,0.000000,0.000000,0.000000,129.181283
4642,2024-01-01,411.850006,412.700012,400.500000,403.950012,309.545959,155897,ASTERDM,HealthCare,303.950012,12.200012,313.554081,29.349773,613.825012,0.000000,0.000000,0.000000,1.128630
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1375,2024-09-12,1619.900024,1649.599976,1606.000000,1643.250000,1643.250000,2356422,TECHM,IT_industry,1543.250000,43.599976,1551.866825,8.905060,2441.525024,3005.294001,2684.340900,51.515936,274.198447
2923,2024-09-12,614.000000,616.450012,608.700012,614.849976,614.849976,1707391,UPL LIMITED,Agriculture,514.849976,7.750000,517.450250,16.166678,921.424988,3005.371000,2678.663150,47.458409,11.960868
343,2024-09-12,520.450012,532.000000,517.750000,530.049988,530.049988,7889737,WIPRO,IT_industry,430.049988,14.250000,434.320534,-0.557801,785.475006,3005.685001,2664.419900,47.376170,-200.396902
6359,2024-09-12,520.450012,532.000000,517.750000,530.049988,530.049988,7889737,WIPRO,IT,430.049988,14.250000,434.320534,46.252827,785.475006,3005.999001,2636.337149,47.934702,-364.490347


# Phase 2: Machine Learning

### Data Preprocessing:

In [66]:
from sklearn.preprocessing import LabelEncoder
label = LabelEncoder()

In [67]:
df.Date = label.fit_transform(df.Date)
df.Symbol = label.fit_transform(df.Symbol)
df.Category = label.fit_transform(df.Category)

In [69]:
df

,Date,Open,High,Low,Close,Adj Close,Volume,Symbol,Category,Daily_Return,Price_Range,Volatility,Cumulative_Return,Average_Price,SMA_50,SMA_200,RSI_14,MACD
4128,0,1598.000000,1620.000000,1585.000000,1598.400024,1598.400024,912757,0,3,1498.400024,35.000000,1520.838340,27.097994,2397.200012,0.000000,0.000000,0.000000,0.000000
2236,0,1318.000000,1336.900024,1310.000000,1320.099976,1320.099976,460112,1,7,1220.099976,26.900024,1237.665116,10.797978,1978.049988,0.000000,0.000000,0.000000,-22.200574
4300,0,5741.000000,5765.000000,5707.850098,5750.049805,5736.050781,127030,2,5,5650.049805,57.149902,5665.733902,29.458001,8616.024902,0.000000,0.000000,0.000000,314.045124
4472,0,179.050003,180.000000,170.050003,175.100006,174.671494,613446,3,5,75.100006,9.949997,82.884068,28.707058,266.600006,0.000000,0.000000,0.000000,129.181283
4642,0,411.850006,412.700012,400.500000,403.950012,309.545959,155897,4,5,303.950012,12.200012,313.554081,29.349773,613.825012,0.000000,0.000000,0.000000,1.128630
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1375,171,1619.900024,1649.599976,1606.000000,1643.250000,1643.250000,2356422,53,7,1543.250000,43.599976,1551.866825,8.905060,2441.525024,3005.294001,2684.340900,51.515936,274.198447
2923,171,614.000000,616.450012,608.700012,614.849976,614.849976,1707391,55,0,514.849976,7.750000,517.450250,16.166678,921.424988,3005.371000,2678.663150,47.458409,11.960868
343,171,520.450012,532.000000,517.750000,530.049988,530.049988,7889737,56,7,430.049988,14.250000,434.320534,-0.557801,785.475006,3005.685001,2664.419900,47.376170,-200.396902
6359,171,520.450012,532.000000,517.750000,530.049988,530.049988,7889737,56,6,430.049988,14.250000,434.320534,46.252827,785.475006,3005.999001,2636.337149,47.934702,-364.490347


### Define Prediction Target & Feature:

In [70]:
x = df.drop('Close', axis=1).values
y = df['Close'].values

### Spliting Data Into Train & Test Set:

In [73]:
from sklearn.model_selection import train_test_split
xtrain,xtest,ytrain,ytest = train_test_split(x,y,test_size=0.2)

### Linear Regression Model:

In [74]:
from sklearn.linear_model import LinearRegression
model = LinearRegression()

In [75]:
model.fit(xtrain,ytrain)

LinearRegression()

In [76]:
yp = model.predict(xtest)
yp

array([2547.55004883, 1645.44995117,  690.04998779, ..., 3031.30004883,
        824.79998779,  813.70001221])

In [93]:
from sklearn.metrics import r2_score
scr = r2_score(ytest,yp)
scr*100

100.0

### Random Forest Regressor:

In [78]:
from sklearn.ensemble import RandomForestRegressor
model = RandomForestRegressor()

In [79]:
model.fit(xtrain,ytrain)

RandomForestRegressor()

In [80]:
ypred = model.predict(xtest)
ypred

array([2548.02902344, 1646.04051392,  690.3385083 , ..., 3031.78253174,
        824.04499512,  814.87749268])

In [81]:
score = r2_score(ytest,ypred)
score*100

99.99903838709128

### XGBoost Regresssor Model:

In [85]:
import xgboost as xgb

In [86]:
model = xgb.XGBRegressor()
model.fit(xtrain,ytrain)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=None, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=None,
             n_jobs=None, num_parallel_tree=None, ...)

In [87]:
ypre = model.predict(xtest)
ypre

array([2550.3806 , 1652.093  ,  690.9283 , ..., 3023.3066 ,  826.25024,
        818.4294 ], dtype=float32)

In [92]:
sc = r2_score(ytest,ypre)
sc*100

99.99087289317596

### Summary:

#### The dataset was first preprocessed to handle any missing values and ensure consistent data types. 
#### Feature engineering was then applied to create additional informative columns, including SMA 50, SMA 200, Relative Strength Index (RSI), and Moving Average Convergence Divergence (MACD), enhancing the dataset for predictive modeling.

#### Categorical features were encoded into numerical values using LabelEncoder, and the dataset was split into features (X) and target (y) variables. 
#### Multiple machine learning models were trained, including Linear Regression, Random Forest Regressor, and XGBoost Regressor, and their performance was evaluated using accuracy metrics.

#### The results provide a comparative understanding of model performance, demonstrating the effectiveness of the preprocessing, feature engineering, and modeling steps in predicting the target variable.